# Version 4 — Validate SQL Before Running It

We add `validate_query(sql)`. The agent must pass validation before `run_query` runs a query.

## Workflow

```text
User Question
     |
     v
AI Data Analyst Agent
     |
     +---- lookup_schema(term) ---------> Pinecone: nlq-schema   (when needed)
     +---- lookup_examples(question) ---> Pinecone: nlq-examples (when useful)
     +---- lookup_common(question) -----> Pinecone: common       (when useful)
     |
     v
Generate read-only SQL
     |
     v
validate_query(sql)
     |
     +---- FAILED ----> correct the SQL ----> validate_query again
     |
     +---- PASSED ----> run_query(sql) ----> PostgreSQL results
                                                  |
                                                  v
                                             Final Answer
```

**Tools:** three lookups, `validate_query(sql)`, `run_query(sql)`.

## What changes from Version 3

| | Version 3 | Version 4 |
|---|---|---|
| SQL check before running | first word only | `validate_query`: SQL policy + PostgreSQL `EXPLAIN` |
| Multiple statements | not detected | rejected |
| Writes hidden in a CTE | not detected | rejected |
| Unknown column / syntax error | found only when the query runs | found by `EXPLAIN`, before running |
| `run_query` safety check | first word | **the same SQL policy** as `validate_query` |
| Prompt | — | “call `validate_query` before `run_query`” |

**Why `run_query` checks too:** the prompt tells the model to validate first, but a prompt is a request, not enforcement. If the model skips validation, `run_query` still refuses unsafe SQL.

## 1. Install dependencies

In [ ]:
%pip install -q langchain langchain-aws langchain-openai pinecone python-dotenv "psycopg[binary]"

## 2. Configuration

| Variable | Required | Used for |
|---|---|---|
| `PINECONE_API_KEY` | yes | Reads the `ecom-kb` index |
| `PINECONE_INDEX_NAME` | no (default `ecom-kb`) | Index name |
| `PINECONE_EMBEDDING_MODEL` | no (default `text-embedding-3-small`) | Must match the model used to fill the index |
| `OPENAI_API_KEY` | yes | Query embeddings only |
| `BEDROCK_CHAT_MODEL_ID` | yes | The chat model (the brain) |
| `AWS_REGION` | no (default `us-east-1`) | Bedrock region; AWS credentials come from your normal AWS config |
| `PGHOST`, `PGDATABASE`, `PGUSER`, `PGPASSWORD` | yes | PostgreSQL connection (use a read-only role) |
| `PGPORT` | no (default `5432`) | |
| `PGSSLMODE` | no | e.g. `require` if your provider needs TLS |

Put these in a `.env` file next to the notebook. Never commit credentials.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Configuration — Version 4
#
#   .env ──► load_dotenv ──► checks ──► model        (Bedrock chat model: the brain)
#                                    ├─► embeddings   (OpenAI: turns a search phrase into a vector)
#                                    ├─► index        (Pinecone `ecom-kb`: the knowledge base)
#                                    └─► PostgreSQL settings (used by the SQL tools)
#
# The embedding model MUST be the same one used when the vectors were
# written to Pinecone, otherwise similarity scores are meaningless.
#
# Does NOT build or upload the knowledge base. It reads the existing index.
# ─────────────────────────────────────────────────────────────
import os
import json
import psycopg
from psycopg.rows import dict_row
from dotenv import load_dotenv

from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_aws import ChatBedrockConverse
from langchain_openai import OpenAIEmbeddings
from pinecone import Pinecone

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY", "")
PINECONE_INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "ecom-kb")
EMBEDDING_MODEL = os.getenv("PINECONE_EMBEDDING_MODEL", "text-embedding-3-small")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
BEDROCK_CHAT_MODEL_ID = os.getenv("BEDROCK_CHAT_MODEL_ID", "")

if not PINECONE_API_KEY:
    raise ValueError("Set PINECONE_API_KEY.")
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Set OPENAI_API_KEY for query embeddings. It must match the embedding model used to build the Pinecone index.")
if not BEDROCK_CHAT_MODEL_ID:
    raise ValueError("Set BEDROCK_CHAT_MODEL_ID to a chat model enabled in your AWS account.")

required_pg_vars = ["PGHOST", "PGDATABASE", "PGUSER", "PGPASSWORD"]
missing_pg_vars = [name for name in required_pg_vars if not os.getenv(name)]
if missing_pg_vars:
    raise ValueError("Missing PostgreSQL environment variables: " + ", ".join(missing_pg_vars))

PGPORT = int(os.getenv("PGPORT", "5432"))
PGSSLMODE = os.getenv("PGSSLMODE")

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
pinecone = Pinecone(api_key=PINECONE_API_KEY)
index = pinecone.Index(PINECONE_INDEX_NAME)
model = ChatBedrockConverse(
    model=BEDROCK_CHAT_MODEL_ID,
    region_name=AWS_REGION,
    temperature=0,
)

## 3. Carried over: knowledge-base tools

These cells are unchanged from Version 3. Run them; the explanation is in Version 3.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Knowledge-base tools — the agent's MEMORY
#
#   lookup_schema(term)        ──► search_namespace("nlq-schema")   ──► tables, columns, joins
#   lookup_examples(question)  ──► search_namespace("nlq-examples") ──► similar question + SQL
#   lookup_common(question)    ──► search_namespace("common")       ──► business definitions
#
#   search_namespace: text ──► embedding ──► Pinecone top-k ──► [{id, score, metadata}]
#   format_matches:   matches ──► readable text (no vectors), cut to max_chars
#
# Does NOT execute SQL and does NOT write to Pinecone.
# ─────────────────────────────────────────────────────────────


def search_namespace(namespace: str, text: str, top_k: int = 8) -> list[dict]:
    """Embed `text` and return the top-k matches from one Pinecone namespace."""
    vector = embeddings.embed_query(text)
    response = index.query(vector=vector, namespace=namespace, top_k=top_k, include_metadata=True)

    # The Pinecone client can return a dict or an object; read both shapes.
    matches = response.get("matches", []) if isinstance(response, dict) else response.matches
    results = []
    for match in matches:
        if isinstance(match, dict):
            results.append({"id": match.get("id", "unknown"),
                            "score": match.get("score"),
                            "metadata": match.get("metadata") or {}})
        else:
            results.append({"id": getattr(match, "id", "unknown"),
                            "score": getattr(match, "score", None),
                            "metadata": getattr(match, "metadata", None) or {}})
    return results


def format_matches(matches: list[dict], max_chars: int = 12000) -> str:
    """Turn matches into text blocks: 'RESULT n | id=... | score=...' plus metadata lines."""
    if not matches:
        return "No matching knowledge-base records were found."
    blocks = []
    for rank, match in enumerate(matches, start=1):
        lines = [f"RESULT {rank} | id={match['id']} | score={match['score']}"]
        for key, value in match["metadata"].items():
            if isinstance(value, (dict, list)):
                value = json.dumps(value, ensure_ascii=False)
            lines.append(f"{key}: {value}")
        blocks.append("\n".join(lines))
    output = "\n\n---\n\n".join(blocks)
    return output[:max_chars] + ("\n...[truncated]" if len(output) > max_chars else "")


@tool
def lookup_schema(term: str) -> str:
    """Search the nlq-schema Pinecone namespace for tables and columns.
    Use it to confirm real table/column names, data types, descriptions, synonyms,
    and business meaning. Retrieves metadata only; does not execute SQL."""
    return format_matches(search_namespace("nlq-schema", term, top_k=8))


@tool
def lookup_examples(question: str) -> str:
    """Search the nlq-examples Pinecone namespace for similar approved NL-to-SQL examples.
    Use examples as patterns, then verify schema details using lookup_schema.
    Retrieves examples only; does not execute SQL."""
    return format_matches(search_namespace("nlq-examples", question, top_k=4), max_chars=10000)


@tool
def lookup_common(question: str) -> str:
    """Search the common Pinecone namespace for shared business definitions or guidance.
    Retrieves knowledge only; does not execute SQL."""
    return format_matches(search_namespace("common", question, top_k=4), max_chars=8000)

## 4. New: one SQL policy

`check_read_only_sql(sql)` is a plain Python function. It does not touch the database, so you can test it right away. Both SQL tools use it.

```text
SELECT ... WHERE status = 'delete requested'     ──► scan copy: WHERE status = ''  ──► allowed
                                                     (the word inside quotes is ignored)
SELECT 1; DROP TABLE orders                      ──► two statements               ──► rejected
WITH x AS (DELETE FROM orders RETURNING *) ...   ──► DELETE inside a CTE          ──► rejected
SELECT * INTO backup FROM orders                 ──► INTO creates a table         ──► rejected
```

In [ ]:
# ─────────────────────────────────────────────────────────────
# check_read_only_sql(sql) — ONE SQL policy, used by both SQL tools
#
#   sql ──► STEP 1 make a "scan copy": remove string literals, then comments
#       ──► STEP 2 exactly one statement?           (a ';' inside a string is fine)
#       ──► STEP 3 starts with SELECT or WITH?
#       ──► STEP 4 no write / admin keyword or risky function anywhere?
#       ──► (True, sql to run)  or  (False, reason)
#
# Why a scan copy: `WHERE status = 'delete requested'` must not be
# rejected because of the word inside the quotes.
#
# Does NOT talk to the database, and is NOT a complete SQL parser.
# The read-only database role is still the real boundary.
# ─────────────────────────────────────────────────────────────
import re

_FORBIDDEN = re.compile(
    r"(?is)\b("
    r"INSERT|UPDATE|DELETE|MERGE|CREATE|ALTER|DROP|TRUNCATE|GRANT|REVOKE|"
    r"CALL|COPY|VACUUM|REFRESH|LOCK|INTO|"          # INTO blocks SELECT ... INTO new_table
    r"pg_sleep|pg_read_file|pg_read_binary_file|pg_ls_dir|lo_import|lo_export|"
    r"dblink\w*|set_config|pg_terminate_backend|pg_cancel_backend"
    r")\b"
)


def _scan_copy(sql: str) -> str:
    sql = re.sub(r"'(?:[^']|'')*'", "''", sql)          # string literals -> ''
    sql = re.sub(r"/\*.*?\*/", " ", sql, flags=re.S)     # /* block comments */
    sql = re.sub(r"--[^\n]*", " ", sql)                  # -- line comments
    return sql.strip()


def check_read_only_sql(sql: str) -> tuple[bool, str]:
    """Return (True, sql_to_run) if the SQL passes the policy, else (False, reason)."""
    if not isinstance(sql, str) or not sql.strip():
        return False, "SQL must be a non-empty string."

    # STEP 1 — scan copy.
    scan = _scan_copy(sql)
    if not scan:
        return False, "SQL is empty after removing comments."

    # STEP 2 — one statement only (one trailing ';' allowed).
    scan = scan[:-1].rstrip() if scan.endswith(";") else scan
    if ";" in scan:
        return False, "Only one SQL statement is allowed."

    # STEP 3 — read-only statement types only.
    if not re.match(r"(?is)^(SELECT|WITH)\b", scan):
        return False, "Only read-only SELECT or WITH queries are allowed."

    # STEP 4 — no writes hidden anywhere (e.g. inside a CTE).
    found = _FORBIDDEN.search(scan)
    if found:
        return False, f"The SQL contains a write or administrative operation ({found.group(0).upper()}) and was rejected."

    # The SQL we run is the ORIGINAL text (literals intact), minus trailing ';'.
    return True, sql.strip().rstrip(";").strip()

In [ ]:
# Test the policy offline — no database needed.
policy_cases = {
    "valid":                     "SELECT COUNT(*) FROM orders",
    "keyword inside a string":   "SELECT COUNT(*) FROM orders WHERE status = 'delete requested'",
    "two statements":            "SELECT 1; SELECT 2",
    "write":                     "DELETE FROM orders",
    "write hidden in a CTE":     "WITH x AS (DELETE FROM orders RETURNING *) SELECT * FROM x",
    "SELECT INTO (creates table)": "SELECT * INTO orders_backup FROM orders",
}
for label, sql in policy_cases.items():
    ok, detail = check_read_only_sql(sql)
    print(f"{'PASS' if ok else 'FAIL'} | {label:28} | {detail}")

## 5. New: `validate_query(sql)`

Two steps: the policy above, then PostgreSQL `EXPLAIN`. `EXPLAIN` parses and plans the query but does not run it, so it catches unknown tables, unknown columns and type errors without reading business data.

In [ ]:
# ─────────────────────────────────────────────────────────────
# PostgreSQL helpers (shared by the SQL tools)
#
#   pg_connect()   ──► new connection, read-only by default, with a statement timeout
#   format_rows()  ──► columns + rows ──► plain-text table the model can read
#
# Does NOT replace a read-only database role. The role is the real
# security boundary; these settings are a second layer.
# ─────────────────────────────────────────────────────────────
MAX_ROWS = 50


def pg_connect(timeout_ms: int = 15000):
    """Open a PostgreSQL connection whose transactions are read-only."""
    kwargs = {
        "host": os.environ["PGHOST"],
        "port": PGPORT,
        "dbname": os.environ["PGDATABASE"],
        "user": os.environ["PGUSER"],
        "password": os.environ["PGPASSWORD"],
        "connect_timeout": 10,
        "row_factory": dict_row,
        "options": f"-c default_transaction_read_only=on -c statement_timeout={timeout_ms}",
    }
    if PGSSLMODE:
        kwargs["sslmode"] = PGSSLMODE
    return psycopg.connect(**kwargs)


def format_rows(columns: list[str], rows: list[dict], truncated: bool) -> str:
    """Render rows as 'col | col' lines. States clearly when rows were cut off."""
    if not rows:
        return "Query succeeded. 0 rows returned."
    header = " | ".join(columns)
    body = "\n".join(" | ".join(str(row.get(c)) for c in columns) for row in rows)
    tail = (f"({len(rows)} rows shown. More rows exist; the result was capped at {MAX_ROWS}.)"
            if truncated else f"({len(rows)} rows.)")
    return f"{header}\n{body}\n{tail}"

In [ ]:
# ─────────────────────────────────────────────────────────────
# validate_query(sql) — check the SQL BEFORE running it
#
#   sql ──► STEP 1 check_read_only_sql (policy, no database)
#       │        └─ fails ──► "VALIDATION FAILED: <reason>"
#       ──► STEP 2 PostgreSQL: EXPLAIN <sql>   (parses + plans, does not run it)
#       │        └─ fails ──► "VALIDATION FAILED: PostgreSQL could not validate ...: <error>"
#       ──► "VALIDATION PASSED ... You may now call run_query(sql)." + plan preview
#
# EXPLAIN catches unknown tables/columns, type errors and syntax errors
# without touching any business data.
#
# Does NOT execute the query and does NOT return business rows.
# ─────────────────────────────────────────────────────────────
@tool
def validate_query(sql: str) -> str:
    """Validate a generated SQL query before execution.
    Checks that it is a single read-only SELECT/WITH statement and asks PostgreSQL
    EXPLAIN to parse and plan it without running the query. Call this before run_query.
    This tool does not execute the query or return business data."""
    # STEP 1 — policy.
    ok, sql_or_reason = check_read_only_sql(sql)
    if not ok:
        return f"VALIDATION FAILED: {sql_or_reason}"

    # STEP 2 — let PostgreSQL parse and plan it.
    try:
        with pg_connect(timeout_ms=10000) as conn:
            plan = conn.execute("EXPLAIN " + sql_or_reason).fetchall()
    except Exception as exc:
        # Short, actionable message; no connection details.
        message = str(exc).splitlines()[0][:500]
        return f"VALIDATION FAILED: PostgreSQL could not validate the query: {message}"

    plan_text = "\n".join(str(row.get("QUERY PLAN", row)) for row in plan[:8])
    return (
        "VALIDATION PASSED: PostgreSQL accepted the query for planning. "
        "The query has not been executed. You may now call run_query(sql) with this exact SQL.\n"
        f"Query plan preview:\n{plan_text[:3000]}"
    )

In [ ]:
# Test validate_query against the real database.
print(validate_query.invoke({"sql": "SELECT COUNT(*) FROM orders"}))
print()
print(validate_query.invoke({"sql": "SELECT no_such_column FROM orders"}))

## 6. Changed: `run_query` uses the same policy

The only change from Version 3: STEP 1 calls `check_read_only_sql` instead of checking the first word.

In [ ]:
# ─────────────────────────────────────────────────────────────
# run_query(sql) — the agent's HANDS (now uses the shared SQL policy)
#
#   sql ──► STEP 1 check_read_only_sql   (same rules as validate_query)
#       ──► STEP 2 execute on a read-only connection
#       ──► STEP 3 format up to 50 rows as text ──► back to the model
#
# Why check again here: the prompt TELLS the model to validate first,
# but a prompt is not enforcement. If the model skips validate_query,
# run_query still refuses unsafe SQL.
#
# Does NOT run EXPLAIN; that is validate_query's job.
# ─────────────────────────────────────────────────────────────
@tool
def run_query(sql: str) -> str:
    """Execute one read-only SELECT or WITH query against PostgreSQL.
    Call validate_query first. Returns at most 50 rows as text, or a message
    starting with "SQL error:" or "REFUSED:" that explains what went wrong."""
    # STEP 1 — policy.
    ok, sql_or_reason = check_read_only_sql(sql)
    if not ok:
        return f"REFUSED: {sql_or_reason}"

    # STEP 2 — execute. Fetch one extra row so we know if the result was cut off.
    try:
        with pg_connect() as conn:
            cursor = conn.execute(sql_or_reason)
            fetched = cursor.fetchmany(MAX_ROWS + 1)
            columns = [d.name for d in cursor.description] if cursor.description else []
    except psycopg.Error as exc:
        return f"SQL error: {exc}"

    # STEP 3 — format for the model.
    return format_rows(columns, fetched[:MAX_ROWS], truncated=len(fetched) > MAX_ROWS)

## 7. The system prompt

New: `validate_query` in **AVAILABLE TOOLS**, and the validate → fix → revalidate → run rules in **SQL WORKFLOW**.

In [ ]:
SYSTEM_PROMPT = """
ROLE
You are an AI Data Analyst Agent for an e-commerce marketplace.

CONTEXT
The existing Pinecone index `ecom-kb` contains `nlq-schema`, `nlq-examples`, and
`common` namespaces. PostgreSQL is the source of truth for internal business data.

AVAILABLE TOOLS
- lookup_schema(term): searches Pinecone `nlq-schema` for schema metadata.
- lookup_examples(question): searches Pinecone `nlq-examples` for similar questions
  and SQL patterns.
- lookup_common(question): searches Pinecone `common` for business definitions.
- validate_query(sql): applies conservative single-statement read-only checks and
  asks PostgreSQL EXPLAIN to parse/plan the query. It does not run the business query.
- run_query(sql): executes a read-only SQL query against PostgreSQL and returns rows.

TOOL-SELECTION AND SQL WORKFLOW
- Identify the business metric, population, filters, date range, and desired grain.
- Retrieve schema details when needed; use examples and common guidance when useful.
- Never copy an example without verifying its schema, joins, filters, and aggregation.
- Generate a single SELECT/WITH query; do not invent columns or relationships.
- Call validate_query(sql) before run_query(sql).
- If validation fails, correct the query using the error and validate it again.
- Call run_query(sql) only after validation reports success.
- Inspect actual query results before answering. If execution fails, correct and retry
  when possible, revalidating each revised query.
- Do not claim validation or execution succeeded unless the corresponding tool confirms it.
- Validation is a guardrail, not a complete security boundary; use a read-only DB role.

CONSTRAINTS
- Never invent numbers, results, schema details, or business definitions.
- Consider table grain, join cardinality, NULLs, and double-counting.
- For percentages, state numerator and denominator.
- If evidence is insufficient, state UNKNOWN and explain why.
- Do not expose private chain-of-thought; provide a concise explanation only.

OUTPUT FORMAT
Use relevant sections:
ANSWER: Answer with units where applicable, or UNKNOWN.
DENOMINATOR: For percentages, identify numerator and denominator; otherwise N/A.
SOURCE: Identify PostgreSQL and include the SQL actually executed, if successful.
CAVEAT: State material limitations, ambiguity, or data-quality concerns.
"""

## 8. Create the agent with five tools

In [ ]:
agent = create_agent(
    model=model,
    tools=[lookup_schema, lookup_examples, lookup_common, validate_query, run_query],
    system_prompt=SYSTEM_PROMPT,
)

## 9. Run the agent and watch validation

In the trace, every `run_query` should follow a `validate_query` that returned **VALIDATION PASSED**. If validation fails, you should see the agent fix the SQL and validate again.

In [ ]:
# ─────────────────────────────────────────────────────────────
# ask_agent(question) — run the agent once and print what happened
#
#   question ──► agent.invoke ──► final state
#                                   │
#                                   ├─ AIMessage with tool_calls ──► "BRAIN → TOOL"
#                                   ├─ ToolMessage               ──► "TOOL → OBSERVATION"
#                                   └─ last message              ──► "FINAL ANSWER"
#
# Does NOT change the agent. It only reads the messages it returns.
# ─────────────────────────────────────────────────────────────
def ask_agent(question: str, agent_to_use=None, max_obs_chars: int = 1500):
    agent_to_use = agent_to_use or agent
    result = agent_to_use.invoke({"messages": [{"role": "user", "content": question}]})

    line = "=" * 80
    print(f"{line}\nQUESTION\n{line}\n{question}")
    print(f"\n{line}\nAGENT TRACE\n{line}")

    for message in result["messages"]:
        kind = type(message).__name__
        if kind == "AIMessage" and getattr(message, "tool_calls", None):
            for call in message.tool_calls:
                print("\n🧠 BRAIN → TOOL")
                print("Tool:", call["name"])
                print("Arguments:", call["args"])
        elif kind == "ToolMessage":
            print(f"\n🖐️ TOOL → OBSERVATION ({message.name})")
            print(str(message.content)[:max_obs_chars])

    print(f"\n{line}\nFINAL ANSWER\n{line}")
    print(result["messages"][-1].content)
    if "sql_call_count" in result:
        print("\nrun_query executions used:", result["sql_call_count"])
    return result

In [ ]:
result = ask_agent("What is the average order value per month this year?")

## 10. Key points

- `validate_query` = SQL policy + `EXPLAIN`. It never runs the query.
- `run_query` enforces the same policy, so skipping validation cannot bypass it.
- Validation errors are written for the model: short, specific, fixable.
- The policy is a guardrail. The read-only database role is still the boundary.

**Next — Version 5:** some questions need facts that are not in our database. We add `web_search` and teach the agent to route between internal data and the web.